In [28]:
# pip install simpy

In [29]:
import simpy

In [30]:
# network bandwidths , server confingurations, and propagation timings
BANDWIDTH_ED1_GW = 20.0  # EdgeDevice1 -> Gateway Link (Mbps)
BANDWIDTH_ED2_GW = 50.0  # EdgeDevice2 -> Gateway Link
BANDWIDTH_GW_ES = 100.0  # Gateway -> EdgeServer Link

SERVER_CPU_SPEED_MHZ = 6000.0  # 2000 MHz
SERVER_CPU_SPEED_GHZ =  SERVER_CPU_SPEED_MHZ / 1000.0 # In  GHZ
SERVER_RAM_MB = 4096 # MB -> total memory available in the server
SERVER_RAM_GB = SERVER_RAM_MB / 1024.0

PROP_DELAY_ED1_GW = 0.005  # 5 ms -> calculation required
PROP_DELAY_ED2_GW = 0.010  # 10 ms
PROP_DELAY_GW_ES = 0.002  # 2 ms

RESPONSE_DELAY = 0.005  # Fixed 5 ms return path delivery delay

In [34]:
# Defining the tasks (in json representation)
TASKS = [
    {
        "id": 1,
        "name": "Task 1 ",
        "source": "EdgeDevice-1",
        "size_mb": 0.2,  # MB
        "cpu_demand": 150.0,  # Megacycles
        "ram_demand": 32,  # MB -> memory demand
        "arrival_time": 0.00,  # seconds
        "deadline": 0.15,  # ms (default unit - seconds)
        "bw_device_gw": BANDWIDTH_ED1_GW,
        "prop_device_gw": PROP_DELAY_ED1_GW,
    },
    {
        "id": 2,
        "name": "Task 2 ",
        "source": "EdgeDevice-2",
        "size_mb": 50.0, # MB
        "cpu_demand": 600.0,  # Megacycles
        "ram_demand": 256,  # MB
        "arrival_time": 0.10,  # seconds
        "deadline": 2.50,  # 2500 ms
        "bw_device_gw": BANDWIDTH_ED2_GW,
        "prop_device_gw": PROP_DELAY_ED2_GW,
    },
    {
        "id": 3,
        "name": "Task 3 ",
        "source": "EdgeDevice-1",
        "size_mb": 0.150, # MB
        "cpu_demand": 5000.0,  # Megacycles
        "ram_demand": 512,  # MB
        "arrival_time": 0.20,  # seconds
        "deadline": 1.80,  # 1800 ms
        "bw_device_gw": BANDWIDTH_ED1_GW,
        "prop_device_gw": PROP_DELAY_ED1_GW,
    },
]

In [35]:
def task_process(env, task, gateway_resource, server_cpu, server_ram):

    task_id = task["id"]
    name = task["name"]
    size_mbits = task["size_mb"] * 8.0  # Convert MegaBytes to MegaBits

    # generating
    yield env.timeout(task["arrival_time"] - env.now)
    t_create = env.now
    print(f"[[{t_create:7.4f}s -> CREATED]]    {name} generated at {task['source']}.")

    # device -> Gateway transmission + propagation
    t_trans_dev_gw = size_mbits / task["bw_device_gw"]
    yield env.timeout(t_trans_dev_gw + task["prop_device_gw"])

    # gateway transfer to Server
    t_gw_arrival = env.now
    print(f"[[{t_gw_arrival:7.4f}s -> GATEWAY TRANSFER]] {name} arrived at Gateway.")

    with gateway_resource.request() as req_gw:
        yield req_gw
        t_gw_queue_end = env.now
        t_gw_queue_delay = t_gw_queue_end - t_gw_arrival

        # gateway -> server transmission + propagation
        t_trans_gw_es = size_mbits / BANDWIDTH_GW_ES
        yield env.timeout(t_trans_gw_es + PROP_DELAY_GW_ES)

    # server arrival & RAM allocation
    t_es_arrival = env.now
    print(f"[[{t_es_arrival:7.4f}s -> ARRIVED EDGE_SERVER ]] {name} arrived at Edge Server. \
    Allocating {task['ram_demand']} MB RAM.")

    yield server_ram.get(task["ram_demand"])

    # CPU Execution
    with server_cpu.request() as req_cpu:
        yield req_cpu
        t_exec_start = env.now
        t_es_queue_delay = t_exec_start - t_es_arrival

        # compute execution duration
        t_exec_duration = task["cpu_demand"] / SERVER_CPU_SPEED_MHZ
        yield env.timeout(t_exec_duration)

    # release RAM back to server memory pool
    yield server_ram.put(task["ram_demand"])

    # response transmission back to origin device
    yield env.timeout(RESPONSE_DELAY)
    t_complete = env.now

    # total end-to-end latency computation
    total_latency = t_complete - t_create
    slack_time = task["deadline"] - total_latency
    status = "SUCCESS" if slack_time >= 0 else "FAIL/TERMINATED"

    print(f"[[{t_complete:7.4f}s -> COMPLETED]]  {name} finished execution.")

    # results (IN JSON format for easy printing)
    task["results"] = {
        "creation_time": t_create,
        "completion_time": t_complete,
        "total_latency": total_latency,
        "dev_gw_trans": t_trans_dev_gw,
        "gw_queue_delay": t_gw_queue_delay,
        "gw_es_trans": t_trans_gw_es,
        "es_queue_delay": t_es_queue_delay,
        "exec_duration": t_exec_duration,
        "slack_time": slack_time,
        "status": status,
    }

In [39]:
def main():
    env = simpy.Environment()

    # Resources
    gateway_resource = simpy.Resource(env, capacity=1) #(Bandwidth)
    server_cpu = simpy.Resource(env, capacity=1) #(CPU Cores)
    server_ram = simpy.Container(env, capacity=SERVER_RAM_MB, init=SERVER_RAM_MB) #(RAM, Disk Storage)

    print("\n")
    print("="*20)
    print(f"CPU Speed: {SERVER_CPU_SPEED_MHZ} MHz ({SERVER_CPU_SPEED_GHZ} GHz)")
    print(f"RAM Capacity: {SERVER_RAM_MB} MB ({SERVER_RAM_GB} GB)")
    print("="*20)
    print("\n")


    # Spawn Task Processes
    for task in TASKS:
        env.process(
            task_process(
                env, task, gateway_resource, server_cpu, server_ram
            )
        )
    print("\n")
    print("="*20)
    print("THREE-TIER EDGE COMPUTING SIMULATION LOG DATA")
    print("="*20)
    print("\n")
    env.run()

    print("\n")
    print("="*20)
    print("LATENCY BREAKDOWN (in seconds)")
    print("="*20)
    header = f"{'Task Name':<22} | {'Device->Gateway':<8} | {'Gateway Queue':<8} | \
    {'Gateway->EdgeServer':<8} | {'EdgeServer Queue':<8} | {'Compute':<8} | {'Total':<8}"
    print(header)
    print("-" * len(header))

    for task in TASKS:
        res = task["results"]
        print(
            f"{task['name']:<22} | "
            f"{res['dev_gw_trans']:8.4f} | "
            f"{res['gw_queue_delay']:8.4f} | "
            f"{res['gw_es_trans']:8.4f} | "
            f"{res['es_queue_delay']:8.4f} | "
            f"{res['exec_duration']:8.4f} | "
            f"{res['total_latency']:8.4f}"
        )

    print("\n")
    print("="*20)
    print("SLA COMPLIANCE SUMMARY ")
    print("="*20)

    sla_header = f"{'Task Name':<22} | {'Deadline':<9} | {'Latency':<9} | {'Slack Margin':<12} | {'SLA Status'}"
    print(sla_header)
    print("-" * len(sla_header))

    for task in TASKS:
        res = task["results"]
        print(
            f"{task['name']:<22} | "
            f"{task['deadline']:7.3f} s | "
            f"{res['total_latency']:7.3f} s | "
            f"{res['slack_time']:+10.4f} s | "
            f"{res['status']}"
        )

if __name__ == "__main__":
    main()



CPU Speed: 6000.0 MHz (6.0 GHz)
RAM Capacity: 4096 MB (4.0 GB)




THREE-TIER EDGE COMPUTING SIMULATION LOG DATA


[[ 0.0000s -> CREATED]]    Task 1  generated at EdgeDevice-1.
[[ 0.0850s -> GATEWAY TRANSFER]] Task 1  arrived at Gateway.
[[ 0.1000s -> CREATED]]    Task 2  generated at EdgeDevice-2.
[[ 0.1030s -> ARRIVED EDGE_SERVER ]] Task 1  arrived at Edge Server.     Allocating 32 MB RAM.
[[ 0.1330s -> COMPLETED]]  Task 1  finished execution.
[[ 0.2000s -> CREATED]]    Task 3  generated at EdgeDevice-1.
[[ 0.2650s -> GATEWAY TRANSFER]] Task 3  arrived at Gateway.
[[ 0.2790s -> ARRIVED EDGE_SERVER ]] Task 3  arrived at Edge Server.     Allocating 512 MB RAM.
[[ 1.1173s -> COMPLETED]]  Task 3  finished execution.
[[ 8.1100s -> GATEWAY TRANSFER]] Task 2  arrived at Gateway.
[[12.1120s -> ARRIVED EDGE_SERVER ]] Task 2  arrived at Edge Server.     Allocating 256 MB RAM.
[[12.2170s -> COMPLETED]]  Task 2  finished execution.


LATENCY BREAKDOWN (in seconds)
Task Name              | Devi